In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive



# Experiment 1
**Baseline**  
*distilbert-base-cased*

In [2]:
def read_conll_2003(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith("-DOCSTART-"):
                continue

            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                splits = line.split()
                if len(splits) >= 4:
                    word = splits[0]
                    pos_tag = splits[1]
                    chunk_tag = splits[2]
                    ner_tag = splits[3]
                    current_sentence.append((word, pos_tag, chunk_tag, ner_tag))
        if current_sentence:
            sentences.append(current_sentence)

    return sentences

file_path = '/drive/MyDrive/Start_LLM/eng.train'
dataset = read_conll_2003(file_path)



In [3]:
import pandas as pd
from datasets import Dataset as HfDataset
from sklearn.model_selection import train_test_split

flat_data = []
for sentence_id, sentence in enumerate(dataset):
    for word, pos, chunk, ner in sentence:
        flat_data.append([sentence_id, word, pos, chunk, ner])

df = pd.DataFrame(flat_data, columns=['Sentence_ID', 'Word', 'POS', 'Chunk', 'NER_Tag'])
grouped = df.groupby("Sentence_ID").agg({
    "Word": list,
    "NER_Tag": list
}).reset_index()
grouped["Full_Sentence"] = grouped["Word"].apply(lambda tokens: " ".join(tokens))

Dataset = grouped[["Sentence_ID", "Full_Sentence", "Word", "NER_Tag"]]
train_dataset = HfDataset.from_pandas(grouped)

dataset_split = train_dataset.train_test_split(test_size=0.2, seed=42)
dataset_split["validation"] = dataset_split["test"]
del dataset_split["test"]

In [4]:
import collections

all_ner_tags = [tag for sublist in train_dataset['NER_Tag'] for tag in sublist]
ner_tag_counts = collections.Counter(all_ner_tags)
ner_tag_counts_df = pd.DataFrame(ner_tag_counts.items(), columns=['NER_Tag', 'Count']).sort_values(by='Count', ascending=False)

label_list = []
label_list = ner_tag_counts_df["NER_Tag"]

In [5]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")


label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["Word"],
        truncation=True,
        batched=False,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:

            label_ids.append(-100)
        elif word_idx != previous_word_idx:

            text_label = examples["NER_Tag"][word_idx]
            label_ids.append(label_to_id[text_label])
        else:
           label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs





config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [6]:
tokenized_dataset = dataset_split.map(
    tokenize_and_align_labels, batched=False
)


Map:   0%|          | 0/11232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['Sentence_ID', 'Word', 'NER_Tag', 'Full_Sentence', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11232
    })
    validation: Dataset({
        features: ['Sentence_ID', 'Word', 'NER_Tag', 'Full_Sentence', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2809
    })
})

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-cased",
    num_labels=len(id_to_label),
    id2label=id_to_label,
    label2id=label_to_id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [9]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5624c6400ac7c63e24cdcd3c22c658f5ee902a4780268c361b694b5cb9e46626
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [12]:
import evaluate
import numpy as np


metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=-1)

    # Remove ignored index (-100) and convert IDs to string labels
    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ner_model_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    metric_for_best_model="f1",

    report_to="none"

)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],


    compute_metrics=compute_metrics,
)


trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.038360,0.051473,0.918392,0.934488,0.926370,0.987055
2,0.024620,0.043121,0.936727,0.948258,0.942457,0.989576
3,0.012010,0.044053,0.939050,0.948258,0.943631,0.990012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2106, training_loss=0.02298969347366014, metrics={'train_runtime': 421.4822, 'train_samples_per_second': 79.946, 'train_steps_per_second': 4.997, 'total_flos': 1100761018785792.0, 'train_loss': 0.02298969347366014, 'epoch': 3.0})

# Experiment 2
**Main** Model

---


*bert-base-cased*

In [14]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["Word"],
        truncation=True,
        batched=False,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:

            label_ids.append(-100)
        elif word_idx != previous_word_idx:

            text_label = examples["NER_Tag"][word_idx]
            label_ids.append(label_to_id[text_label])
        else:
           label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs





config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [15]:
tokenized_dataset = dataset_split.map(
    tokenize_and_align_labels, batched=False
)


Map:   0%|          | 0/11232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [16]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id_to_label),
    id2label=id_to_label,
    label2id=label_to_id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [17]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ner_model_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    metric_for_best_model="f1",

    report_to="none"

)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],


    compute_metrics=compute_metrics,
)


trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.162415,0.042595,0.941103,0.940121,0.940612,0.989624
2,0.043489,0.035837,0.952411,0.956186,0.954295,0.991467
3,0.016153,0.036473,0.951670,0.957229,0.954441,0.991758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2106, training_loss=0.05968474017249213, metrics={'train_runtime': 846.0034, 'train_samples_per_second': 39.83, 'train_steps_per_second': 2.489, 'total_flos': 2201303182860288.0, 'train_loss': 0.05968474017249213, 'epoch': 3.0})

# Experiment 3
**State-of-the-art-ish**

---


*DeBERTa*

In [35]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")


label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["Word"],
        truncation=True,
        batched=False,
        padding=False,

        is_split_into_words=True
    )

    labels = []

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:

            label_ids.append(-100)
        elif word_idx != previous_word_idx:

            text_label = examples["NER_Tag"][word_idx]
            label_ids.append(label_to_id[text_label])
        else:
           label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs





In [36]:
tokenized_dataset = dataset_split.map(
    tokenize_and_align_labels, batched=False
)


Map:   0%|          | 0/11232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [25]:

tokenized_dataset = tokenized_dataset.map(lambda x: {"attention_mask": [int(i) for i in x["attention_mask"]]})


Map:   0%|          | 0/11232 [00:00<?, ? examples/s]

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

In [40]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [47]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=len(id_to_label),
    id2label=id_to_label,
    label2id=label_to_id,
    torch_dtype=torch.float32
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

In [48]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ner_model_results",


    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,



    num_train_epochs=3,

    weight_decay=0.01,

    metric_for_best_model="f1",


    report_to="none"

)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,

    compute_metrics=compute_metrics,
)


trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.137808,0.038073,0.951077,0.949092,0.950084,0.990649
2,0.037978,0.031156,0.960308,0.964114,0.962207,0.992732
3,0.017434,0.030349,0.962186,0.966201,0.964189,0.992975


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2106, training_loss=0.05234076341672501, metrics={'train_runtime': 548.6837, 'train_samples_per_second': 61.412, 'train_steps_per_second': 3.838, 'total_flos': 779323807188576.0, 'train_loss': 0.05234076341672501, 'epoch': 3.0})

In [45]:
import torch

model.eval()

sample = tokenized_dataset["validation"][0]

inputs = {
    "input_ids": torch.tensor([sample["input_ids"]]).to(model.device),
    "attention_mask": torch.tensor([sample["attention_mask"]]).to(model.device),
    "labels": torch.tensor([sample["labels"]]).to(model.device),
}

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.loss)

tensor(nan, device='cuda:0', dtype=torch.float16)


In [46]:
print(next(model.parameters()).dtype)

torch.float16
